## Modified LLM code: single-model training with dynamic landmark evaluation

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

MODEL_NAME = 'Charangan/MedBERT'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Your existing landmark_df loaded
landmark_df = pd.read_csv('landmark_df.csv')

max_visits = 3  # Example for patients with exactly 5 visits
visit_counts = landmark_df['subject_id'].value_counts()
selected_patients = visit_counts[visit_counts == max_visits].index

df_selected = landmark_df[landmark_df['subject_id'].isin(selected_patients)].copy()

results = []
SEED = 42

def narrative_prompt(row):
    narrative = f"Patient is a {row['age_at_landmark']}-year-old {row['gender']}."
    narrative += f" This is the {row['num_total_visits']} visit."
    if row['days_since_previous_visit'] != -1:
        narrative += f" The last visit happened {row['days_since_previous_visit']} days ago."

    if pd.notna(row['diag_text']) and row['diag_text'].strip():
        narrative += f" Medical history includes: {row['diag_text']}."

    if pd.notna(row['med_text']) and row['med_text'].strip():
        narrative += f" Current medications are: {row['med_text']}."

    if pd.notna(row['proc_text']) and row['proc_text'].strip():
        narrative += f" Procedures performed: {row['proc_text']}."

    return narrative

### Discussion on Data Leakage and Methodological Considerations in Landmark-based Fine-Tuning of LLMs

In predictive modeling for longitudinal data, particularly when fine-tuning large language models (LLMs) like MedBERT for dynamic risk prediction at various landmarks, careful methodological considerations are necessary to avoid data leakage.

Initially, we explored an approach where MedBERT would be fine-tuned once on the entire dataset (encompassing all landmarks) and then evaluated dynamically at each landmark visit. However, this raised immediate concerns about potential data leakage. Specifically, data leakage occurs if future information (data collected after a given landmark) inadvertently informs the model's predictions at earlier landmarks, leading to artificially inflated performance metrics and compromised scientific validity.

Upon deeper analysis, we clarified two critical scenarios:

- **Leakage Scenario:** Fine-tuning on data that include information from future landmarks and then evaluating at earlier landmarks inherently leaks future information, compromising evaluation integrity.

- **Leakage-Free Scenario:** Fine-tuning strictly on historical data (data collected before each landmark evaluation) prevents leakage, as the model is only informed by data that would realistically be available at the point of prediction.

Considering this, my current methodological choice involved selecting a subset of patients limited to a specific maximum number of visits (landmarks) and fine-tuning separately at each landmark. This approach explicitly prevents leakage since the model for each landmark only sees historically available data.

Nevertheless, despite this careful methodology, preliminary results showed MedBERT underperforming compared to XGBoost, prompting further exploration into possible reasons:

- **Limited Training Data:** Restricting the dataset to patients with exactly `n` landmark visits substantially reduced the size of the training sets at each landmark, potentially causing overfitting and limiting generalizability.

- **Complexity of MedBERT:** Transformer-based models like MedBERT typically require substantial data and careful hyperparameter tuning. In contrast, XGBoost is robust to smaller datasets and can achieve strong results with relatively straightforward tuning.

To mitigate these issues, several improved, leakage-free strategies were proposed:

1. **Cumulative Historical Training:** Leveraging patients with greater or equal to `n` visits and using cumulative historical data for fine-tuning to significantly enlarge the training set.

2. **Single Historical Model:** Fine-tuning a single model using strictly historical data (e.g., initial landmark visits only) and dynamically evaluating this model across subsequent landmarks. This method capitalizes on larger initial training data and preserves temporal validity.

3. **Incremental Fine-tuning:** Employing transfer learning by initially fine-tuning the model on a large, historically valid dataset and incrementally updating the model with smaller landmark-specific datasets to maintain strong predictive performance over time.

These recommendations highlight essential methodological nuances to ensure robust, leakage-free predictive modeling and suggest viable paths forward for improving MedBERT's performance relative to simpler methods like XGBoost.


